In [1]:
import asyncio
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.model_settings import ModelSettings

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
HOW_MANY_SEARCHES = 3

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def push(message):
    print(f"Push: {message[:100]}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message[:1000]}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)

In [3]:
push("Deep research setup is working")

Push: Deep research setup is working
Status: 200


In [4]:
SEARCH_INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool()],
    model=MODEL_NAME,
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
task = "Most popular AI agent frameworks in 2026"

result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most commonly cited AI agent frameworks are **LangChain/LangGraph, CrewAI, AutoGen, LlamaIndex, OpenAI Agents SDK, Google ADK, Pydantic AI, and Semantic Kernel**. Which one is “most popular” depends on the metric: one 2026 roundup says **LangChain** leads on developer survey use, package downloads, and GitHub stars, while search-trend data puts **LangGraph** ahead in branded search interest. ([drydock.si](https://drydock.si/guides/what-is-the-most-popular-ai-agent-framework?utm_source=openai))

A practical read is that **LangGraph** and **CrewAI** are especially visible in agent discussions, **AutoGen** and **LlamaIndex** remain major community options, and **OpenAI Agents SDK** and **Google ADK** are strong first-party choices. The broader takeaway from recent coverage is that there is no single universal winner; popularity varies by whether you measure surveys, downloads, stars, or search demand. ([drydock.si](https://drydock.si/guides/what-is-the-most-popular-ai-agent-framework?utm_source=openai))

In [6]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [7]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [8]:
PLANNER_INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=PLANNER_INSTRUCTIONS,
    model=MODEL_NAME,
    output_type=WebSearchPlan,
)

In [9]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find current rankings, market adoption, and expert roundups of AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Compare major frameworks and identify which ones are widely used in production today.', query='best AI agent frameworks 2026 LangGraph CrewAI AutoGen review'), WebSearchItem(reason='Capture latest mentions of newer or emerging frameworks and current ecosystem trends.', query='AI agent frameworks 2026 trends popular frameworks')])

In [10]:
for item in result.final_output.searches:
    print("Search:", item.query)
    print("Why:", item.reason)
    print()

Search: most popular AI agent frameworks 2026 ranking
Why: Find current rankings, market adoption, and expert roundups of AI agent frameworks in 2026.

Search: best AI agent frameworks 2026 LangGraph CrewAI AutoGen review
Why: Compare major frameworks and identify which ones are widely used in production today.

Search: AI agent frameworks 2026 trends popular frameworks
Why: Capture latest mentions of newer or emerging frameworks and current ecosystem trends.

